In [3]:
import os, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [6]:
TRAIN_IMG_DIR = "../Dataset/BraTS_2025/glioma split data/train/images"
TRAIN_MASK_DIR = "../Dataset/BraTS_2025/glioma split data/train/masks"

VAL_IMG_DIR = "../Dataset/BraTS_2025/glioma split data/val/images"
VAL_MASK_DIR = "../Dataset/BraTS_2025/glioma split data/val/masks"

In [7]:
train_images = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.npy")))
train_masks  = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, "*.npy")))

val_images = sorted(glob.glob(os.path.join(VAL_IMG_DIR, "*.npy")))
val_masks  = sorted(glob.glob(os.path.join(VAL_MASK_DIR, "*.npy")))

In [8]:
def normalize(volume):
    return (volume - np.mean(volume)) / (np.std(volume) + 1e-8)

In [9]:
def extract_patch(img, mask, patch_size=64, max_tries=10):
    h, w, d = img.shape[:3]

    for _ in range(max_tries):
        x = np.random.randint(0, h - patch_size)
        y = np.random.randint(0, w - patch_size)
        z = np.random.randint(0, d - patch_size)

        img_patch = img[x:x+patch_size, y:y+patch_size, z:z+patch_size]
        mask_patch = mask[x:x+patch_size, y:y+patch_size, z:z+patch_size]

        if np.sum(mask_patch) > 0:
            return img_patch, mask_patch

    return img_patch, mask_patch

In [10]:
class BrainDataset(Dataset):
    def __init__(self, image_files, mask_files):
        self.image_files = image_files
        self.mask_files = mask_files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img = np.load(self.image_files[idx])
        mask = np.load(self.mask_files[idx])

        img = normalize(img)
        img, mask = extract_patch(img, mask)

        # (H,W,D,C) → (C,H,W,D)
        img = np.transpose(img, (3,0,1,2))
        mask = np.transpose(mask, (3,0,1,2))

        return torch.tensor(img, dtype=torch.float32), \
               torch.tensor(mask, dtype=torch.float32)

In [11]:
train_loader = DataLoader(BrainDataset(train_images, train_masks),
                          batch_size=2, shuffle=True)

val_loader = DataLoader(BrainDataset(val_images, val_masks),
                        batch_size=2)

In [12]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.c1 = ConvBlock(4, 32)
        self.p1 = nn.MaxPool3d(2)

        self.c2 = ConvBlock(32, 64)
        self.p2 = nn.MaxPool3d(2)

        self.c3 = ConvBlock(64, 128)
        self.p3 = nn.MaxPool3d(2)

        self.c4 = ConvBlock(128, 256)

        self.u1 = nn.ConvTranspose3d(256, 128, 2, stride=2)
        self.c5 = ConvBlock(256, 128)

        self.u2 = nn.ConvTranspose3d(128, 64, 2, stride=2)
        self.c6 = ConvBlock(128, 64)

        self.u3 = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.c7 = ConvBlock(64, 32)

        self.out = nn.Conv3d(32, 4, 1)

    def forward(self, x):
        c1 = self.c1(x)
        c2 = self.c2(self.p1(c1))
        c3 = self.c3(self.p2(c2))
        c4 = self.c4(self.p3(c3))

        u1 = self.u1(c4)
        u1 = torch.cat([u1, c3], dim=1)
        c5 = self.c5(u1)

        u2 = self.u2(c5)
        u2 = torch.cat([u2, c2], dim=1)
        c6 = self.c6(u2)

        u3 = self.u3(c6)
        u3 = torch.cat([u3, c1], dim=1)
        c7 = self.c7(u3)

        return torch.sigmoid(self.out(c7))

In [13]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = y_true.view(-1, 4)
    y_pred = y_pred.view(-1, 4)

    intersection = (y_true * y_pred).sum()
    denom = y_true.sum() + y_pred.sum()

    return 1 - (2. * intersection + smooth) / (denom + smooth)


def combined_loss(y_true, y_pred):
    bce = F.binary_cross_entropy(y_pred, y_true)
    d = dice_loss(y_true, y_pred)
    return bce + d

In [14]:
model = UNet3D().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 30

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x)

        loss = combined_loss(y, preds)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {train_loss/len(train_loader):.4f}")

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "brats_unet.pth")

NameError: name 'model' is not defined

In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)

            loss = combined_loss(y, preds)
            total_loss += loss.item()

    return total_loss / len(loader)